# Movie Search Engine: Live Demo
**Author:** Aslı Akel  
**Programme:** MSc Data Science and Artificial Intelligence  

This notebook demonstrates the full search pipeline interactively.  


## 1. Install Dependencies
Run once, then restart runtime. Do not run again

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import subprocess
subprocess.run(
    ['pip', 'install', '-q',
     'transformers==4.46.2',
     'langchain==0.1.17',
     'langchain-core==0.1.53',
     'ragatouille==0.0.9'],
    capture_output=True
)
print('Done. Now go to Runtime -> Restart session, then start from Cell 2.')

Done. Now go to Runtime -> Restart session, then start from Cell 2.


## 2. Load Pipeline


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
plt.ioff()
plt.show = lambda *args, **kwargs: None

import os
from contextlib import redirect_stdout, redirect_stderr

with open(os.devnull, "w") as f:
    with redirect_stdout(f), redirect_stderr(f):
        %run movie_search_engine_pipeline.py

print(" Pipeline loaded successfully.")
print(f"Documents: {len(prepared_df):,}")
print(f"Vocabulary size: {len(inverted_index):,}")
print("Available methods: overview | baseline | linear | freqcomb")
print(f"ColBERT enabled: {RUN_COLBERT}")

artifact.metadata: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

 Pipeline loaded successfully.
Documents: 4,799
Vocabulary size: 35,212
Available methods: overview | baseline | linear | freqcomb
ColBERT enabled: True


## Score Adjustment Layer

In [ ]:
def score_adjust(bm25_sc, vote_avg, vote_count, popularity, alpha=0.1):
    if bm25_sc <= 0:
        return 0.0
    vote_signal = (vote_avg / 10.0) if vote_avg else 0.5
    confidence = min(vote_count / 50, 1.0) if vote_count else 0.0
    pop_signal = min(math.log1p(popularity) / 10.0, 1.0) if popularity else 0.0
    metadata_signal = (vote_signal * confidence + pop_signal) / 2.0
    return bm25_sc * (1.0 + alpha * metadata_signal)

In [ ]:
def rank_results(query, method='freqcomb', top_k=10):
    dispatch = {
        'overview': search_overview,
        'baseline': search_baseline,
        'linear': search_linear,
        'freqcomb': search_freqcomb,
    }

    if method not in dispatch:
        raise ValueError(f"Unknown method '{method}'. Choose: {list(dispatch.keys())}")

    raw = dispatch[method](query, top_k=top_k)

    output = []
    for doc_id, bm25_sc in raw:
        row = prepared_df.loc[doc_id]
        adj_score = score_adjust(
            bm25_sc,
            row['vote_average'],
            row['vote_count'],
            row['popularity']
        )
        ov = str(row.get('overview', ''))
        output.append({
            'rank': 0,
            'doc_id': int(doc_id),
            'title': row['title_raw'],
            'score': round(float(adj_score), 4),
            'overview_snippet': ov[:150] + '...' if len(ov) > 150 else ov,
        })

    output.sort(key=lambda x: x['score'], reverse=True)
    output = output[:top_k]

    for i, r in enumerate(output, 1):
        r['rank'] = i

    return output

## 3. Live Search
Change QUERY and METHOD to try different searches.
Available methods: overview · baseline · linear · freqcomb

In [19]:
QUERY  = 'artificial intelligence robot'
METHOD = 'freqcomb'
TOP_K  = 10

results = rank_results(QUERY, method=METHOD, top_k=TOP_K)

print(f'Query  : "{QUERY}"')
print(f'Method : {METHOD}\n')
print(f'{"Rank":<6} {"Title":<45} {"Score"}')
print('-' * 65)
for r in results:
    print(f'{r["rank"]:<6} {r["title"]:<45} {r["score"]}')

Query  : "artificial intelligence robot"
Method : freqcomb

Rank   Title                                         Score
-----------------------------------------------------------------
1      I, Robot                                      22.1081
2      Ex Machina                                    22.0174
3      Chappie                                       20.8908
4      Automata                                      20.3744
5      A.I. Artificial Intelligence                  20.3511
6      Terminator Salvation                          18.5094
7      Terminator Genisys                            17.0127
8      The Terminator                                16.6824
9      Stealth                                       15.9946
10     Terminator 3: Rise of the Machines            15.959


## 4. Method Comparison
Shows how the same query performs across all 4 retrieval methods.

In [20]:
QUERY = 'artificial intelligence robot'

print(f'Query: "{QUERY}"\n')
print(f'{"Method":<12} {"Rank 1":<35} {"Rank 2":<35} {"Rank 3"}')
print('-' * 100)

for method in ['overview', 'baseline', 'linear', 'freqcomb']:
    res = rank_results(QUERY, method=method, top_k=3)
    r1  = res[0]['title'] if len(res) > 0 else 'n/a'
    r2  = res[1]['title'] if len(res) > 1 else 'n/a'
    r3  = res[2]['title'] if len(res) > 2 else 'n/a'
    print(f'{method:<12} {r1:<35} {r2:<35} {r3}')

Query: "artificial intelligence robot"

Method       Rank 1                              Rank 2                              Rank 3
----------------------------------------------------------------------------------------------------
overview     Ex Machina                          Stealth                             Repo Men
baseline     I, Robot                            Ex Machina                          Chappie
linear       A.I. Artificial Intelligence        I, Robot                            Robots
freqcomb     I, Robot                            Ex Machina                          Chappie


## 5. Index Sample Data
Vocabulary size, average document length, and sample IDF values.


In [21]:
print('=== Inverted Index Stats ===')
print(f'Total documents    : {len(prepared_df):,}')
print(f'Vocabulary size    : {len(inverted_index):,} unique terms')
print(f'Avg doc length     : {avgdl:.1f} tokens')

print('\n=== Sample IDF Values ===')
print(f'{"Term":<15} {"Doc frequency":<18} {"IDF score"}')
print('-' * 45)
for term in ['robot', 'artificial', 'intelligence', 'action', 'love']:
    df_val = len(inverted_index.get(term, []))
    idf    = idf_store.get(term, 0.0)
    print(f'{term:<15} {df_val:<18} {idf:.4f}')

=== Inverted Index Stats ===
Total documents    : 4,799
Vocabulary size    : 35,212 unique terms
Avg doc length     : 110.5 tokens

=== Sample IDF Values ===
Term            Doc frequency      IDF score
---------------------------------------------
robot           46                 4.6369
artificial      32                 4.9951
intelligence    82                 4.0636
action          1228               1.3628
love            806                1.7837


## 6. BM25 Scoring Breakdown
Shows exactly how a document is scored for a given query — term by term.

In [22]:
QUERY = 'artificial intelligence robot'
MOVIE_TITLE = 'I, Robot'

print("Note: this is the core BM25 score before the lightweight metadata-based score adjustment is applied.\n")

q_terms = normalise(QUERY).split()
match = prepared_df[prepared_df['title_raw'] == MOVIE_TITLE]

if match.empty:
    print(f'Movie "{MOVIE_TITLE}" not found.')
else:
    doc_id = match.index[0]
    dl = weighted_doc_len[doc_id]

    print(f'Query : "{QUERY}" -> normalised terms: {q_terms}')
    print(f'Movie : {MOVIE_TITLE}')
    print(f'Doc length (weighted): {dl:.1f} | Avg: {avg_weighted_dl:.1f}')
    print(f'{"Term":<15} {"Agg TF":<12} {"IDF":<10} {"BM25 contribution"}')
    print('-' * 55)

    total = 0.0
    for term in q_terms:
        agg_tf = sum(FIELD_WEIGHTS[f] * field_tf[f][doc_id].get(term, 0) for f in FIELDS)
        idf_v = global_idf.get(term, 0.0)
        contrib = bm25_score(agg_tf, idf_v, dl, avg_weighted_dl) if agg_tf > 0 else 0.0
        total += contrib
        print(f'{term:<15} {agg_tf:<12.1f} {idf_v:<10.4f} {contrib:.4f}')

    print(f'\nTotal BM25 score: {total:.4f}')

Note: this is the core BM25 score before the lightweight metadata-based score adjustment is applied.

Query : "artificial intelligence robot" -> normalised terms: ['artificial', 'intelligence', 'robot']
Movie : I, Robot
Doc length (weighted): 98.0 | Avg: 95.2
Term            Agg TF       IDF        BM25 contribution
-------------------------------------------------------
artificial      1.5          4.9951     6.1760
intelligence    1.5          4.0636     5.0242
robot           8.0          4.6369     9.7282

Total BM25 score: 20.9284


## 7. ColBERT Re-ranking
Shows BM25 FreqComb top-5 vs ColBERT re-ranked top-5 side by side.

In [25]:
QUERY = 'artificial intelligence robot'

bm25_res    = rank_results(QUERY, method='freqcomb', top_k=10)
colbert_res = rerank_with_colbert(QUERY, bm25_top_k=COLBERT_BM25_TOP_K, final_top_k=50)

print(f'Query: "{QUERY}"\n')
print(f'{"Rank":<6} {"BM25 FreqComb":<40} {"ColBERT Re-ranked"}')
print('-' * 85)
for i in range(10):
    b = bm25_res[i]['title']    if i < len(bm25_res)    else 'n/a'
    c = colbert_res[i]['title'] if i < len(colbert_res) else 'n/a'
    print(f'{i+1:<6} {b:<40} {c}')

Query: "artificial intelligence robot"

Rank   BM25 FreqComb                            ColBERT Re-ranked
-------------------------------------------------------------------------------------
1      I, Robot                                 I, Robot
2      Ex Machina                               Automata
3      Chappie                                  A.I. Artificial Intelligence
4      Automata                                 Chappie
5      A.I. Artificial Intelligence             Ex Machina
6      Terminator Salvation                     Terminator Salvation
7      Terminator Genisys                       The Terminator
8      The Terminator                           Terminator 3: Rise of the Machines
9      Stealth                                  Terminator Genisys
10     Terminator 3: Rise of the Machines       Interstellar


## 8. Evaluation Results
Full system comparison across all 5 configurations.

In [24]:
import pandas as pd

full_summary = pd.read_csv('full_system_comparison.csv', index_col=0)
print('=== Full System Comparison — Mean Metrics Across 30 Queries ===\n')
print(full_summary.to_string())

print('\n=== Best system per metric ===')
for col in full_summary.columns:
    best = full_summary[col].idxmax()
    print(f'  {col:<12}: {best} ({full_summary.loc[best, col]:.4f})')

=== Full System Comparison — Mean Metrics Across 30 Queries ===

                                    P@5    P@10  Recall@10   F1@10     MAP  NDCG@10
Configuration                                                                      
Overview-only BM25               0.3200  0.2533     0.1788  0.2080  0.1259   0.2432
Baseline BM25 (weighted_text)    0.6400  0.5833     0.4177  0.4805  0.3435   0.5382
Linear Field-Score Combination   0.5067  0.4733     0.3440  0.3929  0.2460   0.4158
FreqComb                         0.6800  0.5900     0.4211  0.4851  0.3527   0.5372
ColBERT Re-ranker (BM25 top-50)  0.6333  0.5933     0.4262  0.4900  0.3470   0.5338

=== Best system per metric ===
  P@5         : FreqComb (0.6800)
  P@10        : ColBERT Re-ranker (BM25 top-50) (0.5933)
  Recall@10   : ColBERT Re-ranker (BM25 top-50) (0.4262)
  F1@10       : ColBERT Re-ranker (BM25 top-50) (0.4900)
  MAP         : FreqComb (0.3527)
  NDCG@10     : Baseline BM25 (weighted_text) (0.5382)
